In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🎯 Domain-Driven Design & Event Sourcing Guide

**Building scalable systems with DDD principles and event sourcing architecture**

This guide covers:
- Domain-driven design principles
- Bounded contexts and ubiquitous language
- Aggregates and value objects
- Event sourcing fundamentals
- Event store implementation
- Projections and read models
- CQRS patterns
- Event versioning and migration
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Domain-Driven Design

### Bounded Contexts & Aggregates
```python
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set
from datetime import datetime
from enum import Enum
from abc import ABC, abstractmethod

class ValueObject:
    """Base class for value objects"""
    
    def __eq__(self, other):
        if not isinstance(other, self.__class__):
            return False
        return self.__dict__ == other.__dict__
    
    def __hash__(self):
        return hash(frozenset(self.__dict__.items()))

@dataclass
class Money(ValueObject):
    """Money value object"""
    amount: float
    currency: str
    
    def __post_init__(self):
        if self.amount < 0:
            raise ValueError("Amount cannot be negative")

@dataclass
class OrderId(ValueObject):
    """Order ID value object"""
    value: str

@dataclass
class CustomerId(ValueObject):
    """Customer ID value object"""
    value: str

class OrderStatus(Enum):
    """Order status"""
    PENDING = "pending"
    CONFIRMED = "confirmed"
    SHIPPED = "shipped"
    DELIVERED = "delivered"
    CANCELLED = "cancelled"

@dataclass
class OrderItem:
    """Order item"""
    product_id: str
    quantity: int
    unit_price: Money
    
    def get_total(self) -> Money:
        """Calculate total for item"""
        return Money(
            self.unit_price.amount * self.quantity,
            self.unit_price.currency
        )

class Order:
    """Order aggregate"""
    
    def __init__(self, order_id: OrderId, customer_id: CustomerId):
        self.order_id = order_id
        self.customer_id = customer_id
        self.items: List[OrderItem] = []
        self.status = OrderStatus.PENDING
        self.created_at = datetime.now()
        self.updated_at = datetime.now()
        self.version = 0
        self.uncommitted_events: List[Dict] = []
    
    def add_item(self, item: OrderItem):
        """Add item to order"""
        
        if self.status != OrderStatus.PENDING:
            raise ValueError(f"Cannot add items to {self.status.value} order")
        
        self.items.append(item)
        
        self.uncommitted_events.append({
            'event_type': 'ItemAdded',
            'aggregate_id': str(self.order_id.value),
            'timestamp': datetime.now(),
            'data': {
                'product_id': item.product_id,
                'quantity': item.quantity,
                'unit_price': item.unit_price.amount
            }
        })
        
        self.updated_at = datetime.now()
    
    def confirm_order(self) -> bool:
        """Confirm order"""
        
        if not self.items:
            raise ValueError("Cannot confirm empty order")
        
        if self.status != OrderStatus.PENDING:
            raise ValueError(f"Order is {self.status.value}")
        
        self.status = OrderStatus.CONFIRMED
        
        self.uncommitted_events.append({
            'event_type': 'OrderConfirmed',
            'aggregate_id': str(self.order_id.value),
            'timestamp': datetime.now(),
            'data': {'total_items': len(self.items)}
        })
        
        self.updated_at = datetime.now()
        
        return True
    
    def get_total(self) -> Money:
        """Calculate order total"""
        
        total = 0.0
        currency = "USD"
        
        for item in self.items:
            total += item.get_total().amount
            currency = item.unit_price.currency
        
        return Money(total, currency)
    
    def get_uncommitted_events(self) -> List[Dict]:
        """Get uncommitted events"""
        return self.uncommitted_events
    
    def mark_events_as_committed(self):
        """Mark events as committed"""
        self.uncommitted_events.clear()
        self.version += 1
```

### Bounded Context Pattern
```python
class BoundedContext:
    """Bounded context for domain"""
    
    def __init__(self, name: str):
        self.name = name
        self.aggregates: Dict[str, object] = {}
        self.entities: Dict[str, object] = {}
        self.value_objects: Set[type] = set()
        self.services: Dict[str, object] = {}
    
    def register_aggregate(self, aggregate_type: type):
        """Register aggregate type"""
        self.aggregates[aggregate_type.__name__] = aggregate_type
    
    def register_value_object(self, vo_type: type):
        """Register value object type"""
        self.value_objects.add(vo_type)
    
    def register_service(self, service_name: str, service: object):
        """Register domain service"""
        self.services[service_name] = service
    
    def get_context_map(self) -> Dict:
        """Get context map"""
        
        return {
            'name': self.name,
            'aggregates': list(self.aggregates.keys()),
            'value_objects': [vo.__name__ for vo in self.value_objects],
            'services': list(self.services.keys())
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Event Sourcing

### Event Store
```python
@dataclass
class Event:
    """Domain event"""
    event_id: str
    aggregate_id: str
    aggregate_type: str
    event_type: str
    version: int
    timestamp: datetime
    data: Dict
    metadata: Dict = field(default_factory=dict)

class EventStore:
    """Store domain events"""
    
    def __init__(self):
        self.events: List[Event] = []
        self.snapshots: Dict[str, Dict] = {}
        self.event_handlers: Dict[str, List[callable]] = {}
    
    def append(self, event: Event) -> bool:
        """Append event to store"""
        
        self.events.append(event)
        
        # Trigger handlers
        if event.event_type in self.event_handlers:
            for handler in self.event_handlers[event.event_type]:
                handler(event)
        
        return True
    
    def get_events(self, aggregate_id: str, from_version: int = 0) -> List[Event]:
        """Get events for aggregate"""
        
        return [
            e for e in self.events
            if e.aggregate_id == aggregate_id and e.version >= from_version
        ]
    
    def subscribe(self, event_type: str, handler: callable):
        """Subscribe to event type"""
        
        if event_type not in self.event_handlers:
            self.event_handlers[event_type] = []
        
        self.event_handlers[event_type].append(handler)
    
    def create_snapshot(self, aggregate_id: str, version: int, state: Dict):
        """Create snapshot for fast replay"""
        
        self.snapshots[f"{aggregate_id}@{version}"] = {
            'aggregate_id': aggregate_id,
            'version': version,
            'state': state,
            'created_at': datetime.now()
        }
    
    def get_snapshot(self, aggregate_id: str, before_version: int) -> Optional[Dict]:
        """Get latest snapshot before version"""
        
        matching = [
            snap for snap in self.snapshots.values()
            if snap['aggregate_id'] == aggregate_id and snap['version'] <= before_version
        ]
        
        if not matching:
            return None
        
        return max(matching, key=lambda x: x['version'])
    
    def get_event_store_stats(self) -> Dict:
        """Get event store statistics"""
        
        return {
            'total_events': len(self.events),
            'total_snapshots': len(self.snapshots),
            'event_types': list(set(e.event_type for e in self.events)),
            'earliest_event': self.events[0].timestamp if self.events else None,
            'latest_event': self.events[-1].timestamp if self.events else None
        }
```

### Projections & Read Models
```python
class Projection:
    """Read model projection"""
    
    def __init__(self, name: str):
        self.name = name
        self.state: Dict = {}
        self.version = 0
    
    def handle_event(self, event: Event):
        """Update projection with event"""
        raise NotImplementedError

class OrderProjection(Projection):
    """Projection for order statistics"""
    
    def __init__(self):
        super().__init__("OrderProjection")
        self.state = {
            'total_orders': 0,
            'total_revenue': 0.0,
            'orders_by_status': {},
            'customer_stats': {}
        }
    
    def handle_event(self, event: Event):
        """Update projection"""
        
        if event.event_type == 'OrderConfirmed':
            self.state['total_orders'] += 1
            customer_id = event.data.get('customer_id')
            
            if customer_id:
                if customer_id not in self.state['customer_stats']:
                    self.state['customer_stats'][customer_id] = {'order_count': 0, 'total_spent': 0}
                
                self.state['customer_stats'][customer_id]['order_count'] += 1
        
        elif event.event_type == 'ItemAdded':
            total_price = event.data.get('unit_price', 0) * event.data.get('quantity', 0)
            self.state['total_revenue'] += total_price
        
        self.version += 1
    
    def get_top_customers(self, limit: int = 5) -> List[Dict]:
        """Get top customers by order count"""
        
        customers = self.state.get('customer_stats', {})
        
        sorted_customers = sorted(
            customers.items(),
            key=lambda x: x[1]['order_count'],
            reverse=True
        )
        
        return [
            {'customer_id': cid, 'order_count': data['order_count']}
            for cid, data in sorted_customers[:limit]
        ]

class ProjectionManager:
    """Manage multiple projections"""
    
    def __init__(self):
        self.projections: Dict[str, Projection] = {}
    
    def register_projection(self, projection: Projection):
        """Register projection"""
        self.projections[projection.name] = projection
    
    def handle_event(self, event: Event):
        """Handle event for all projections"""
        
        for projection in self.projections.values():
            projection.handle_event(event)
    
    def rebuild_all(self, events: List[Event]):
        """Rebuild all projections from events"""
        
        for projection in self.projections.values():
            projection.state.clear()
            projection.version = 0
        
        for event in events:
            self.handle_event(event)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. CQRS Pattern

### Command Query Responsibility Segregation
```python
class Command:
    """Base command class"""
    pass

@dataclass
class CreateOrderCommand(Command):
    """Create order command"""
    order_id: str
    customer_id: str

@dataclass
class AddItemToOrderCommand(Command):
    """Add item to order command"""
    order_id: str
    product_id: str
    quantity: int
    unit_price: float

class CommandBus:
    """Route commands to handlers"""
    
    def __init__(self):
        self.handlers: Dict[type, callable] = {}
    
    def register_handler(self, command_type: type, handler: callable):
        """Register command handler"""
        self.handlers[command_type] = handler
    
    def execute(self, command: Command) -> bool:
        """Execute command"""
        
        handler = self.handlers.get(type(command))
        
        if not handler:
            raise ValueError(f"No handler for {type(command).__name__}")
        
        return handler(command)

class QueryBus:
    """Route queries to handlers"""
    
    def __init__(self):
        self.handlers: Dict[type, callable] = {}
    
    def register_handler(self, query_type: type, handler: callable):
        """Register query handler"""
        self.handlers[query_type] = handler
    
    def execute(self, query) -> any:
        """Execute query"""
        
        handler = self.handlers.get(type(query))
        
        if not handler:
            raise ValueError(f"No handler for {type(query).__name__}")
        
        return handler(query)

class CQRSApplication:
    """Application with CQRS"""
    
    def __init__(self):
        self.command_bus = CommandBus()
        self.query_bus = QueryBus()
        self.event_store = EventStore()
        self.projection_manager = ProjectionManager()
    
    def execute_command(self, command: Command) -> bool:
        """Execute command and emit events"""
        return self.command_bus.execute(command)
    
    def execute_query(self, query) -> any:
        """Execute query on projections"""
        return self.query_bus.execute(query)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Domain-Driven Design & Event Sourcing Checklist

✅ **Domain Design**
- [ ] Bounded contexts identified
- [ ] Ubiquitous language defined
- [ ] Aggregates designed
- [ ] Value objects created
- [ ] Domain services identified

✅ **Event Sourcing**
- [ ] Event store implemented
- [ ] Events designed
- [ ] Event versioning planned
- [ ] Snapshots configured
- [ ] Replay tested

✅ **Projections**
- [ ] Projections created
- [ ] Read models optimized
- [ ] Query performance verified
- [ ] Eventual consistency acceptable
- [ ] Rebuild strategy

✅ **CQRS Implementation**
- [ ] Commands defined
- [ ] Queries defined
- [ ] Buses implemented
- [ ] Handlers registered
- [ ] Separation maintained

✅ **Operational**
- [ ] Event versioning strategy
- [ ] Migration plan
- [ ] Audit trail enabled
- [ ] Replay tested
- [ ] Team trained
</VSCode.Cell>
```